# TMDB Emotion Intensity Pipeline

Downloads the TMDb movies dataset (~930K movies) via `kagglehub`, extracts movie keywords, and scores them using the NRC Emotion Intensity Lexicon to produce per-movie emotion intensity features (anger, fear, joy, sadness, disgust, surprise, anticipation, trust).

Ensure `.env` has `KAGGLE_API_TOKEN` (or `~/.kaggle` credentials).

In [1]:
import os
import zipfile
from pathlib import Path

import kagglehub
import pandas as pd
from dotenv import load_dotenv

# Load .env so KAGGLE_API_TOKEN (e.g. KGAT_...) is available in the notebook
load_dotenv()

# Optional: fail fast with a clear message if Kaggle credentials are missing
def _has_kaggle_credentials() -> bool:
    if os.environ.get("KAGGLE_API_TOKEN") or (os.environ.get("KAGGLE_USERNAME") and os.environ.get("KAGGLE_KEY")):
        return True
    kaggle_dir = Path.home() / ".kaggle"
    return any((kaggle_dir / name).exists() for name in ("kaggle.json", "access_token", "access_token.txt"))

if not _has_kaggle_credentials():
    raise RuntimeError(
        "Kaggle credentials not found. Set KAGGLE_API_TOKEN in .env (or KAGGLE_USERNAME + KAGGLE_KEY), "
        "or add kaggle.json to ~/.kaggle. See readme.md."
    )

DATASET = "asaniczka/tmdb-movies-dataset-2023-930k-movies"
NROWS = 10_000  # set to None to load full dataset

# Download full dataset (no file path) — single-file path causes 404 for this dataset
try:
    base_path = kagglehub.dataset_download(DATASET)
except Exception as e:
    err = str(e).lower()
    if "404" in err or "not found" in err:
        raise RuntimeError(
            f"Dataset not found (404). Check that it exists: {DATASET}"
        ) from e
    if "401" in err or "403" in err or "unauthorized" in err or "forbidden" in err:
        raise RuntimeError(
            "Kaggle returned 401/403. Check credentials and accept the dataset terms on the dataset page."
        ) from e
    raise

# Resolve path: kagglehub returns a dir (extracted) or a zip file
pandas_kwargs = {"nrows": NROWS} if NROWS else {}
base = Path(base_path)
if base.is_file() and base.suffix.lower() == ".zip":
    with zipfile.ZipFile(base, "r") as z:
        csv_names = sorted(n for n in z.namelist() if n.lower().endswith(".csv"))
        if not csv_names:
            raise FileNotFoundError(f"No .csv in {base}")
        with z.open(csv_names[0]) as f:
            df = pd.read_csv(f, **pandas_kwargs)
else:
    zips = list(base.rglob("*.zip"))
    if zips:
        with zipfile.ZipFile(sorted(zips)[0], "r") as z:
            csv_names = sorted(n for n in z.namelist() if n.lower().endswith(".csv"))
            if not csv_names:
                raise FileNotFoundError(f"No .csv in {zips[0]}")
            with z.open(csv_names[0]) as f:
                df = pd.read_csv(f, **pandas_kwargs)
    else:
        csvs = list(base.rglob("*.csv"))
        if not csvs:
            raise FileNotFoundError(f"No .zip or .csv under {base_path}")
        df = pd.read_csv(sorted(csvs)[0], **pandas_kwargs)
print(f"Loaded {len(df)} rows × {len(df.columns)} columns.")

Loaded 10000 rows × 24 columns.


In [8]:
df.head(10)

,id,title,vote_average,vote_count,status,release_date,revenue,runtime,adult,backdrop_path,...,original_title,overview,popularity,poster_path,tagline,genres,production_companies,production_countries,spoken_languages,keywords
0,27205,Inception,8.364,34495,Released,2010-07-15,825532764,148,False,/8ZTVqvKDQ8emSGUEMjsS4yHAwrp.jpg,...,Inception,"Cobb, a skilled thief who commits corporate es...",83.952,/oYuLEt3zVCKq57qu2F8dT7NIa6f.jpg,Your mind is the scene of the crime.,"Action, Science Fiction, Adventure","Legendary Pictures, Syncopy, Warner Bros. Pict...","United Kingdom, United States of America","English, French, Japanese, Swahili","rescue, mission, dream, airplane, paris, franc..."
1,157336,Interstellar,8.417,32571,Released,2014-11-05,701729206,169,False,/pbrkL804c8yAv3zBZR4QPEafpAR.jpg,...,Interstellar,The adventures of a group of explorers who mak...,140.241,/gEU2QniE6E77NI6lCU6MxlNBvIx.jpg,Mankind was born on Earth. It was never meant ...,"Adventure, Drama, Science Fiction","Legendary Pictures, Syncopy, Lynda Obst Produc...","United Kingdom, United States of America",English,"rescue, future, spacecraft, race against time,..."
2,155,The Dark Knight,8.512,30619,Released,2008-07-16,1004558444,152,False,/nMKdUUepR0i5zn0y1T4CsSB5chy.jpg,...,The Dark Knight,Batman raises the stakes in his war on crime. ...,130.643,/qJ2tW6WMUDux911r6m7haRef0WH.jpg,Welcome to a world without rules.,"Drama, Action, Crime, Thriller","DC Comics, Legendary Pictures, Syncopy, Isobel...","United Kingdom, United States of America","English, Mandarin","joker, sadism, chaos, secret identity, crime f..."
3,19995,Avatar,7.573,29815,Released,2009-12-15,2923706026,162,False,/vL5LR6WdxWPjLPFRLe133jXWsh5.jpg,...,Avatar,"In the 22nd century, a paraplegic Marine is di...",79.932,/kyeqWdyUXW608qlYkRqosgbbJyK.jpg,Enter the world of Pandora.,"Action, Adventure, Fantasy, Science Fiction","Dune Entertainment, Lightstorm Entertainment, ...","United States of America, United Kingdom","English, Spanish","future, society, culture clash, space travel, ..."
4,24428,The Avengers,7.710,29166,Released,2012-04-25,1518815515,143,False,/9BBTo63ANSmhC4e6r62OJFuK2GL.jpg,...,The Avengers,When an unexpected enemy emerges and threatens...,98.082,/RYMX2wcKCBAr24UyPD7xwmjaTn.jpg,Some assembly required.,"Science Fiction, Action, Adventure",Marvel Studios,United States of America,"English, Hindi, Russian","new york city, superhero, shield, based on com..."
5,293660,Deadpool,7.606,28894,Released,2016-02-09,783100000,108,False,/en971MEXui9diirXlogOrPKmsEn.jpg,...,Deadpool,The origin story of former Special Forces oper...,72.735,/zq8Cl3PNIDGU3iWNRoc5nEZ6pCe.jpg,Witness the beginning of a happy ending.,"Action, Adventure, Comedy","20th Century Fox, The Donners' Company, Genre ...",United States of America,English,"superhero, anti hero, mercenary, based on comi..."
6,299536,Avengers: Infinity War,8.255,27713,Released,2018-04-25,2052415039,149,False,/mDfJG3LC3Dqb67AZ52x3Z0jU0uB.jpg,...,Avengers: Infinity War,As the Avengers and their allies have continue...,154.340,/7WsyChQLEftFiDOVTGkv3hFpyyt.jpg,An entire universe. Once and for all.,"Adventure, Action, Science Fiction",Marvel Studios,United States of America,"English, Xhosa","sacrifice, magic, superhero, based on comic, s..."
7,550,Fight Club,8.438,27238,Released,1999-10-15,100853753,139,False,/hZkgoQYus5vegHoetLkCJzb17zJ.jpg,...,Fight Club,A ticking-time-bomb insomniac and a slippery s...,69.498,/pB8BM7pdSp6B6Ih7QZ4DrQ3PmJK.jpg,Mischief. Mayhem. Soap.,Drama,"Regency Enterprises, Fox 2000 Pictures, Taurus...",United States of America,English,"dual identity, rage and hate, based on novel o..."
8,118340,Guardians of the Galaxy,7.906,26638,Released,2014-07-30,772776600,121,False,/uLtVbjvS1O7gXL8lUOwsFOH4man.jpg,...,Guardians of the Galaxy,"Light years from Earth, 26 years after being a...",33.255,/r7vmZjiyZw9rpJMQJdXpjgiCOk9.jpg,All heroes start somewhere.,"Action, Science Fiction, Adventure",Marvel Studios,United States of America,English,"spacec

In [ ]:
# All unique TMDB keywords
keywords_series = df["keywords"].dropna().astype(str)
unique_keywords = sorted(
    {kw.strip() for s in keywords_series for kw in s.split(",") if kw.strip()}
)
print(f"Unique TMDB keywords in sample: {len(unique_keywords):,}")
print(unique_keywords[:50])

## NRC Word-Emotion Association Lexicon

[NRC Emotion Lexicon](https://www.saifmohammad.com/WebPages/NRC-Emotion-Lexicon.htm) v0.92: words × associations with eight emotions (anger, fear, anticipation, trust, surprise, sadness, joy, disgust) and two sentiments (negative, positive). File: `data/NRC-emotion-lexicon-wordlevel-alphabetized-v0.92.txt` (tab-separated: word, emotion, 0|1). Refresh from source: [NRC-Emotion-Lexicon.zip](https://www.saifmohammad.com/WebDocs/Lexicons/NRC-Emotion-Lexicon.zip).

In [9]:
import pandas as pd
from pathlib import Path

NRC_PATH = Path("data/NRC-emotion-lexicon-wordlevel-alphabetized-v0.92.txt")
# Data starts after 46 lines of header/comments
nrc_long = pd.read_csv(NRC_PATH, sep="\t", header=None, names=["word", "emotion", "value"], skiprows=46)
print(f"NRC lexicon: {len(nrc_long):,} rows (word × emotion).")
print(nrc_long.head(15))

NRC lexicon: 141,820 rows (word × emotion).
      word       emotion  value
0    aback         anger      0
1    aback  anticipation      0
2    aback       disgust      0
3    aback          fear      0
4    aback           joy      0
5    aback      negative      0
6    aback      positive      0
7    aback       sadness      0
8    aback      surprise      0
9    aback         trust      0
10  abacus         anger      0
11  abacus  anticipation      0
12  abacus       disgust      0
13  abacus          fear      0
14  abacus           joy      0


In [10]:
# Pivot: one row per word, columns = emotions/sentiments (0/1)
nrc_wide = nrc_long.pivot(index="word", columns="emotion", values="value").reset_index()
print(f"Unique words: {len(nrc_wide):,}. Emotions/sentiments: {list(nrc_wide.columns[1:])}")
nrc_wide.head(10)

Unique words: 14,182. Emotions/sentiments: ['anger', 'anticipation', 'disgust', 'fear', 'joy', 'negative', 'positive', 'sadness', 'surprise', 'trust']


emotion,word,anger,anticipation,disgust,fear,joy,negative,positive,sadness,surprise,trust
0,NaN,0,0,0,0,0,0,0,0,0,0
1,aback,0,0,0,0,0,0,0,0,0,0
2,abacus,0,0,0,0,0,0,0,0,0,1
3,abandon,0,0,0,1,0,1,0,1,0,0
4,abandoned,1,0,0,1,0,1,0,1,0,0
5,abandonment,1,0,0,1,0,1,0,1,1,0
6,abate,0,0,0,0,0,0,0,0,0,0
7,abatement,0,0,0,0,0,0,0,0,0,0
8,abba,0,0,0,0,0,0,1,0,0,0
9,abbot,0,0,0,0,0,0,0,0,0,1


## NRC Emotion Intensity Lexicon

[NRC Emotion Intensity Lexicon](https://www.saifmohammad.com/WebPages/AffectIntensity.htm) v1: real-valued scores (0–1) for eight emotions. [Download zip](https://www.saifmohammad.com/WebDocs/Lexicons/NRC-Emotion-Intensity-Lexicon.zip); we load the English file `NRC-Emotion-Intensity-Lexicon-v1.txt` (word, emotion, intensity).

In [2]:
import urllib.request
import zipfile
from pathlib import Path

NRC_INTENSITY_ZIP = "https://www.saifmohammad.com/WebDocs/Lexicons/NRC-Emotion-Intensity-Lexicon.zip"
DATA_DIR = Path("data")
ZIP_PATH = DATA_DIR / "NRC-Emotion-Intensity-Lexicon.zip"
EXTRACT_DIR = DATA_DIR / "NRC-Emotion-Intensity-Lexicon"
INTENSITY_FILE = EXTRACT_DIR / "NRC-Emotion-Intensity-Lexicon-v1.txt"

if not INTENSITY_FILE.exists():
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    req = urllib.request.Request(
        NRC_INTENSITY_ZIP,
        headers={
            "User-Agent": "Mozilla/5.0 (compatible; Python)",
            "Accept": "*/*",
        },
    )
    with urllib.request.urlopen(req, timeout=60) as resp, open(ZIP_PATH, "wb") as out:
        out.write(resp.read())
    with zipfile.ZipFile(ZIP_PATH, "r") as z:
        z.extractall(DATA_DIR)

nrc_intensity_long = pd.read_csv(
    INTENSITY_FILE, sep="\t", header=None, names=["word", "emotion", "intensity"]
)
print(f"NRC intensity: {len(nrc_intensity_long):,} rows. Emotions: {nrc_intensity_long['emotion'].unique().tolist()}")
nrc_intensity_long.head(10)

NRC intensity: 9,829 rows. Emotions: ['anger', 'anticipation', 'disgust', 'fear', 'joy', 'sadness', 'surprise', 'trust']


,word,emotion,intensity
0,outraged,anger,0.964
1,brutality,anger,0.959
2,hatred,anger,0.953
3,hateful,anger,0.940
4,terrorize,anger,0.939
5,violently,anger,0.938
6,infuriated,anger,0.938
7,furious,anger,0.929
8,furiously,anger,0.927
9,enraged,anger,0.927


In [ ]:
# Pivot: one row per word, columns = emotion intensity scores (0–1)
# NaN → 0: lexicon only records non-zero entries; missing = no signal
nrc_intensity_wide = (
    nrc_intensity_long
    .pivot(index="word", columns="emotion", values="intensity")
    .fillna(0)
    .reset_index()
)
print(f"Words with intensity scores: {len(nrc_intensity_wide):,}")
nrc_intensity_wide.head(10)

## Joined Lexicon: binary presence + continuous intensity + sentiment polarity

NRC EmoLex (binary 0/1 per emotion + positive/negative sentiment) and NRC Emotion Intensity Lexicon (real-valued 0–1) share the same word-level ontology. Joining on `word` gives each term: **which emotions it activates**, **how strongly**, and **its overall polarity**.

In [ ]:
EMOTIONS = ["anger", "anticipation", "disgust", "fear", "joy", "sadness", "surprise", "trust"]

# Rename intensity columns to avoid collision with binary columns
intensity_renamed = nrc_intensity_wide.rename(
    columns={e: f"{e}_intensity" for e in EMOTIONS}
)

# Outer join: keep words from either lexicon
nrc_joined = nrc_wide.merge(intensity_renamed, on="word", how="outer").fillna(0)

# Cast binary columns back to int
binary_cols = EMOTIONS + ["positive", "negative"]
nrc_joined[binary_cols] = nrc_joined[binary_cols].astype(int)

print(f"Joined lexicon: {len(nrc_joined):,} words")
print(f"Columns: {list(nrc_joined.columns)}")
nrc_joined.head(10)

In [ ]:
# Emotion–sentiment polarity mapping (derived from NRC EmoLex, computed empirically).
# For each emotion, % of words with that emotion also labelled positive/negative:
#
#   emotion        pos%    neg%    signal
#   ─────────────────────────────────────────────────────
#   joy           98.4%    5.4%    strongly positive
#   trust         69.3%    5.8%    positive
#   anticipation  57.5%   21.1%    positive
#   surprise      41.4%   40.8%    mixed
#   anger          4.7%   92.0%    strongly negative
#   disgust        2.8%   91.8%    strongly negative
#   sadness        4.8%   89.7%    strongly negative
#   fear           6.8%   83.4%    negative
#   ─────────────────────────────────────────────────────
#
# Polarity-weighted intensity:
#   {emotion}_positive_intensity = intensity * positive_flag
#   {emotion}_negative_intensity = intensity * negative_flag
#
# Summary scores:
#   mean_positive_intensity = mean({joy,trust,anticipation,surprise}_positive_intensity)
#   mean_negative_intensity = mean({anger,disgust,sadness,fear}_negative_intensity)

EMOTIONS = ["anger", "anticipation", "disgust", "fear", "joy", "sadness", "surprise", "trust"]

nrc_scored = nrc_joined.copy()

pos_cols, neg_cols = [], []
for e in EMOTIONS:
    pos_col = f"{e}_positive_intensity"
    neg_col = f"{e}_negative_intensity"
    nrc_scored[pos_col] = nrc_scored[f"{e}_intensity"] * nrc_scored["positive"]
    nrc_scored[neg_col] = nrc_scored[f"{e}_intensity"] * nrc_scored["negative"]
    pos_cols.append(pos_col)
    neg_cols.append(neg_col)

nrc_scored["mean_positive_intensity"] = nrc_scored[pos_cols].mean(axis=1)
nrc_scored["mean_negative_intensity"] = nrc_scored[neg_cols].mean(axis=1)

display_cols = ["word"] + pos_cols + neg_cols + ["mean_positive_intensity", "mean_negative_intensity"]
print(f"Columns added: {len(pos_cols + neg_cols) + 2}")

examples = ["rage", "celebrate", "betrayal", "wedding", "murder", "hope", "surprise"]
mask = nrc_scored["word"].isin(examples)
nrc_scored[mask][["word"] + [f"{e}_intensity" for e in EMOTIONS] + ["mean_positive_intensity", "mean_negative_intensity"]].set_index("word")

In [ ]:
# Keyword coverage against the joined lexicon
nrc_words = set(nrc_joined["word"])
direct_matches = [kw for kw in unique_keywords if kw in nrc_words]
token_matches = [
    kw for kw in unique_keywords
    if any(tok in nrc_words for tok in kw.split())
]
print(f"Total unique TMDB keywords:     {len(unique_keywords):>6,}")
print(f"Direct matches (joined lexicon):{len(direct_matches):>6,} ({100*len(direct_matches)/len(unique_keywords):.1f}%)")
print(f"At least one token matches:     {len(token_matches):>6,} ({100*len(token_matches)/len(unique_keywords):.1f}%)")
print(f"\nSample unmatched:")
print([kw for kw in unique_keywords if kw not in nrc_words][:20])

In [ ]:
df.info()